<a href="https://colab.research.google.com/github/AmnonElias/DeepLearningProject/blob/main/smart_pest_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install ultralytics
from google.colab import drive
drive.mount('/content/drive')

import cv2
import os
from ultralytics import YOLO # ייבוא מודל רשת הנוירונים

def run_smart_pest_detection(video_source, camera_type='RGB', output_name='smart_output.avi', target_fps=2):
    # 1. טעינת רשת הנוירונים המאומנת (ה"מוח")
    model_path = '/content/drive/My Drive/DeepLearning/TrainedModel/pest_detection_nano_yolo26n_epoch50/weights/best.pt'
    print(f"טוען את מודל ה-YOLO מהנתיב: {model_path}...")
    model = YOLO(model_path)

    # 2. הגדרות מצלמה אדפטיביות (כמו שעשינו קודם)
    if camera_type == 'IR':
        VAR_THRESH = 16
        MIN_AREA = 20
        USE_MORPHOLOGY = True
    else: # RGB
        VAR_THRESH = 50
        MIN_AREA = 500
        USE_MORPHOLOGY = False

    # 3. פתיחת הסרטון והגדרת שמירת הפלט
    cap = cv2.VideoCapture(video_source)
    if not cap.isOpened():
        print("שגיאה בפתיחת הסרטון.")
        return

    original_fps = cap.get(cv2.CAP_PROP_FPS)
    if original_fps == 0: original_fps = 30
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*'DIVX')
    out = cv2.VideoWriter(output_name, fourcc, target_fps, (width, height))
    frame_skip_interval = int(original_fps / target_fps)

    backSub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=VAR_THRESH, detectShadows=False)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    frame_count = 0

    print("מתחיל בעיבוד הסרטון החכם...")

    while True:
        ret, frame = cap.read()
        if not ret: break

        frame_count += 1

        # שלב 1: דילוג על פריימים (חיסכון במשאבים)
        if frame_count % frame_skip_interval != 0:
            continue

        # שלב 2: עיבוד מקדים לזיהוי תנועה
        if camera_type == 'IR':
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            processed_frame = cv2.GaussianBlur(gray, (5, 5), 0)
        else:
            processed_frame = frame

        # שלב 3: מזהה התנועה ה"טיפש" והמהיר
        fgMask = backSub.apply(processed_frame)
        _, fgMask = cv2.threshold(fgMask, 200, 255, cv2.THRESH_BINARY)

        if USE_MORPHOLOGY:
            fgMask = cv2.dilate(fgMask, kernel, iterations=2)

        contours, _ = cv2.findContours(fgMask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        motion_detected = False

        for contour in contours:
            if cv2.contourArea(contour) > MIN_AREA:
                motion_detected = True
                break # מספיק שמצאנו אזור אחד שזז כדי להעיר את הרשת

        # ====================================================
        # שלב 4: הקסם! הפעלת הרשת רק אם זיהינו תנועה
        # ====================================================
        if motion_detected:
            # מעבירים את הפריים למודל ה-YOLO.
            # conf=0.4 אומר: תציג רק זיהויים שהרשת בטוחה בהם ב-40% ומעלה (אפשר לשנות)
            results = model(frame, conf=0.3, verbose=False)

            # הפונקציה plot() מציירת אוטומטית את הריבועים והשמות של החיות על התמונה!
            annotated_frame = results[0].plot()

            # נוסיף גם כיתוב קטן שלנו כדי לדעת שהרשת פעלה בפריים הזה
            cv2.putText(annotated_frame, "YOLO ACTIVATED", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

            out.write(annotated_frame)
        else:
            # אם אין תנועה, אנחנו פשוט כותבים את הפריים הרגיל לסרטון בלי להפעיל את YOLO
            out.write(frame)

    cap.release()
    out.release()
    print(f"העיבוד הסתיים בהצלחה! הסרטון המנותח נשמר בשם: {output_name}")

# דוגמה להפעלה (אל תשכח לעדכן את הנתיבים לקבצים שלך):
run_smart_pest_detection('/content/drive/My Drive/DeepLearning/Videos/davinci_a_snake_in_a_yard_outside_a_house__simulating_IR.mp4', camera_type='IR', output_name='/content/drive/My Drive/DeepLearning/Videos/davinci_a_snake_in_a_yard_outside_a_house__simulating_IR_yolo.avi', target_fps=4)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
טוען את מודל ה-YOLO מהנתיב: /content/drive/My Drive/DeepLearning/TrainedModel/pest_detection_nano_yolo26n_epoch50/weights/best.pt...
מתחיל בעיבוד הסרטון החכם...
העיבוד הסתיים בהצלחה! הסרטון המנותח נשמר בשם: /content/drive/My Drive/DeepLearning/Videos/davinci_a_snake_in_a_yard_outside_a_house__simulating_IR_yolo.avi
